# ⭐ Self-Correction Demo

This tutorial demonstrates the core concept of **ResonanceFlow**: end-to-end self-correction of protein structures using differentiable physics and NMR constraints.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elkins/resonance-flow/blob/main/examples/interactive_tutorials/self_correction_demo.ipynb)

In [ ]:
# Setup: Install ResonanceFlow, Matplotlib, and Py3Dmol
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q resonance-flow matplotlib py3Dmol biotite jax jaxlib optax flax
else:
    import os

    sys.path.insert(0, os.path.abspath("../../"))

## 1. Setting up the Optimization Loop

We will initialize a random Transformer model and generate an initial random coordinate trace for a 10-residue sequence. Then, we will use JAX and Optax to optimize the weights of the model *through* the differentiable physics engine to minimize steric clashes, bond length violations, and RDC errors!

In [ ]:
import jax
import jax.numpy as jnp

from resonance_flow.model import TransformerCoordinatePredictor
from resonance_flow.train import create_train_state, train_step

rng = jax.random.PRNGKey(42)
rng, init_rng = jax.random.split(rng)

model = TransformerCoordinatePredictor(vocab_size=21, d_model=32, num_heads=2, num_layers=2)
state = create_train_state(init_rng, model, learning_rate=0.01)

batch_size = 1
seq_len = 20
batch = jax.random.randint(rng, (batch_size, seq_len), 0, 21)
atom_radii = jnp.ones((seq_len,)) * 1.5
measured_rdcs = jnp.zeros((seq_len - 2,))  # Dummy experimental RDCs

initial_coords = state.apply_fn(
    {"params": state.params},
    batch,
    deterministic=True,
    rngs={"dropout": rng},
)[0]

print("Setup complete. Initial structure generated.")

## 2. Running the Differentiable Training Loop

Let's run 150 epochs of optimization and track how the loss terms fall as the protein self-corrects.

In [ ]:
num_steps = 150
history = {"total": [], "steric": [], "bond": [], "rdc": []}

for _ in range(num_steps):
    rng, dropout_rng = jax.random.split(rng)
    state, loss, l_steric, l_bond, l_rdc = train_step(
        state, batch, dropout_rng, atom_radii, measured_rdcs
    )
    history["total"].append(float(loss))
    history["steric"].append(float(l_steric))
    history["bond"].append(float(l_bond))
    history["rdc"].append(float(l_rdc))

print(f"Finished {num_steps} steps!")

final_coords = state.apply_fn(
    {"params": state.params},
    batch,
    deterministic=True,
    rngs={"dropout": rng},
)[0]

## 3. Visualizing the Loss Trajectory

We can see how the optimizer quickly minimized steric clashes and bond length violations.

In [ ]:
import matplotlib.pyplot as plt

epochs = range(num_steps)

plt.figure(figsize=(10, 5))
plt.plot(epochs, history["total"], "k-", lw=2, label="Total Loss")
plt.plot(epochs, history["bond"], "g--", lw=2, label="Bond Penalty")
plt.plot(epochs, history["steric"], "r--", lw=2, label="Steric Clash")
plt.plot(epochs, history["rdc"], "b--", lw=2, label="RDC Error")

plt.title("ResonanceFlow Self-Correction Trajectory")
plt.xlabel("Optimization Step")
plt.ylabel("Loss")
plt.yscale("log")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Visualizing Structural Refinement

Let's export the initial random coil and the final corrected structure to compare them in Py3Dmol. You will notice that the final structure is much more extended and "protein-like" because the steric and bond constraints forced it out of its collapsed random state!

In [ ]:
import biotite.structure as struc
import biotite.structure.io.pdb as pdb
import numpy as np
import py3Dmol


def array_to_pdb(coords_jnp, filename):
    coords = np.array(coords_jnp)
    L = coords.shape[0]
    array = struc.AtomArray(L)
    array.coord = coords
    array.atom_name = ["CA"] * L
    array.element = ["C"] * L
    array.res_id = np.arange(1, L + 1)
    array.res_name = ["GLY"] * L
    array.chain_id = ["A"] * L
    pdb_file = pdb.PDBFile()
    pdb.set_structure(pdb_file, array)
    pdb_file.write(filename)


array_to_pdb(initial_coords, "initial.pdb")
array_to_pdb(final_coords, "final.pdb")

with open("initial.pdb") as f:
    init_str = f.read()
with open("final.pdb") as f:
    final_str = f.read()

view = py3Dmol.view(width=800, height=400, linked=False, viewergrid=(1, 2))
view.addModel(init_str, "pdb", viewer=(0, 0))
view.setStyle({"model": -1}, {"line": {"color": "red", "linewidth": 5}}, viewer=(0, 0))
view.addModel(final_str, "pdb", viewer=(0, 1))
view.setStyle({"model": -1}, {"line": {"color": "green", "linewidth": 5}}, viewer=(0, 1))
view.zoomTo()
view.show()